In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# ==========================================
# 1. DEFINE THE NEURAL NETWORK ARCHITECTURE
# ==========================================
class HousePricePredictor(nn.Module):
    def __init__(self):
        super(HousePricePredictor, self).__init__()
        
        # INPUT LAYER: Takes 3 raw features (SqFt, Bedrooms, Age)
        # Passes them to 16 neurons in the first hidden layer.
        self.input_layer = nn.Linear(in_features=3, out_features=16)
        
        # HIDDEN LAYERS: Extracts complex patterns from features
        self.hidden_layer_1 = nn.Linear(in_features=16, out_features=8)
        self.hidden_layer_2 = nn.Linear(in_features=8, out_features=4)
        
        # Activation Function (adds non-linearity so the network can learn)
        self.relu = nn.ReLU()
        
        # OUTPUT LAYER: Condenses patterns down to 1 final price prediction
        self.output_layer = nn.Linear(in_features=4, out_features=1)

    def forward(self, x):
        # Pass data through input layer + activation
        x = self.relu(self.input_layer(x))   
        
        # Pass data through hidden layers + activations
        x = self.relu(self.hidden_layer_1(x)) 
        x = self.relu(self.hidden_layer_2(x)) 
        
        # Pass through output layer (no activation for regression/continuous values)
        x = self.output_layer(x)              
        return x


# ==========================================
# 2. GENERATE SAMPLE DATA (Normalized)
# ==========================================
# Inputs [SqFt (in thousands), Bedrooms, Age (in decades)]
X_train = torch.tensor([
    [1.5, 2.0, 1.0],  # House 1
    [2.0, 3.0, 0.5],  # House 2
    [2.5, 4.0, 1.5],  # House 3
    [1.2, 2.0, 2.0],  # House 4
    [3.0, 4.0, 0.2]   # House 5
], dtype=torch.float32)

# Actual Prices (in hundreds of thousands of dollars)
y_train = torch.tensor([
    [250.0],
    [320.0],
    [380.0],
    [190.0],
    [450.0]
], dtype=torch.float32)


# ==========================================
# 3. INITIALIZE MODEL, LOSS & OPTIMIZER
# ==========================================
model = HousePricePredictor()

# Loss Function: Mean Squared Error (calculates how far off predictions are)
criterion = nn.MSELoss()

# Optimizer: Adam (adjusts internal weights to reduce error)
optimizer = optim.Adam(model.parameters(), lr=0.01)


# ==========================================
# 4. TRAINING LOOP
# ==========================================
epochs = 200

print("--- Starting Training ---")
for epoch in range(1, epochs + 1):
    # Step A: Forward pass (make predictions)
    predictions = model(X_train)
    
    # Step B: Calculate loss (measure error)
    loss = criterion(predictions, y_train)
    
    # Step C: Backward pass (calculate gradients)
    optimizer.zero_grad()  # Reset old gradients
    loss.backward()        # Backpropagation
    
    # Step D: Update network weights
    optimizer.step()
    
    # Print progress every 40 epochs
    if epoch % 40 == 0:
        print(f"Epoch [{epoch}/{epochs}] - Loss (Error): {loss.item():.4f}")


# ==========================================
# 5. TEST ON NEW UNSEEN DATA
# ==========================================
model.eval()  # Set model to evaluation mode

# A new house: 2,200 sqft (2.2), 3 beds (3.0), 8 years old (0.8)
new_house = torch.tensor([[2.2, 3.0, 0.8]], dtype=torch.float32)

with torch.no_grad():  # Turn off gradient tracking for evaluation
    predicted_price = model(new_house)

print("\n--- Model Evaluation ---")
print(f"Predicted Price for New House: ${predicted_price.item() * 1000:,.2f}")